# 08: Representational Inferential Analysis

## Overview
Within-phase hypothesis testing for Phase 3 (Representational). Aggregates statistical
tests from NB01-NB06 (base model) and NB02c-NB06c (circuit), applies **BH-FDR correction**
across all tests, and produces verdict tables.

## Hypothesis Domains
- **R1**: Embedding Structure (6 tests): from NB01
- **R2**: Residual Stream Dynamics (5 tests): from NB02
- **R3**: Attention Patterns (5 tests): from NB04
- **R4**: MLP Contributions (4 tests): from NB05
- **R5**: Information-Theoretic (4 tests): from NB06
- **RC1**: Circuit vs Base Representations (5 tests): from NB02c-NB06c
- **RC2**: Circuit Band-Dependence (4 tests): from NB02c-NB06c
- **RC3**: Circuit Scaling (3 tests): from NB02c-NB06c

**Excluded** (moved to Phase 4 Integration):
- R6: Representation-Structure links
- R7: Representation-Function links
- R8: Cross-phase variance decomposition

## Notebook Structure
1. Setup & Data Loading
2. R1: Embedding Structure Tests
3. R2: Residual Stream Tests
4. R3: Attention Pattern Tests
5. R4: MLP Tests
6. R5: Information-Theoretic Tests
7. RC1: Circuit vs Base Representations
8. RC2: Circuit Band-Dependence
9. RC3: Circuit Scaling
10. BH-FDR Correction & Summary
11. Visualizations
12. Export

## Data Sources
- Base analysis CSVs: `outputs/{domain}/base/analysis/`
- Circuit analysis CSVs: `outputs/{domain}/circuit/analysis/`
- Comparison CSVs: `outputs/{domain}/comparison/analysis/`

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from functools import partial as _partial

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_CAPACITY,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    RANDOM_SEED,
    ALPHA,
    LOW_FREQ_BANDS,
    HIGH_FREQ_BANDS,
    get_domain_dirs,
)
from utils.data_loading import (
    save_analysis,
    load_domain_csv,
)
from utils.plotting import setup_plotting, save_figure

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

setup_plotting()
np.random.seed(RANDOM_SEED)

ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("inferential", "inferential")
save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

# Collect all test results
all_results = []


def add_result(
    domain,
    hypothesis,
    model,
    comparison,
    stat_name,
    stat_value,
    p_value,
    effect_size=None,
    n=None,
):
    all_results.append(
        {
            "domain": domain,
            "hypothesis": hypothesis,
            "model": model,
            "comparison": comparison,
            "stat_name": stat_name,
            "stat_value": float(stat_value) if stat_value is not None else np.nan,
            "p_value": float(p_value) if p_value is not None else np.nan,
            "effect_size": float(effect_size) if effect_size is not None else np.nan,
            "n": int(n) if n is not None else np.nan,
        }
    )


print(f"Output: {ANALYSIS_DIR}")

Output: LSC_circuit_analysis/03_Phase_Representational/outputs/inferential/analysis


In [2]:
# Load all analysis CSVs into a dict for easy access
def safe_load(domain, variant, filename):
    try:
        return load_domain_csv(domain, variant, filename)
    except FileNotFoundError:
        return pd.DataFrame()


data = {
    # NB01 Embedding
    "knn": safe_load("embedding", "base", "01_knn_purity.csv"),
    "emb_master": safe_load("embedding", "base", "01_master_embedding.csv"),
    "norms": safe_load("embedding", "base", "01_embedding_norms.csv"),
    "cka": safe_load("embedding", "base", "01_cka_matrices.csv"),
    "dim": safe_load("embedding", "base", "01_dimensionality.csv"),
    # NB02 Residual
    "probe": safe_load("residual_stream", "base", "02_probe_trajectory.csv"),
    "separation": safe_load("residual_stream", "base", "02_separation_trajectory.csv"),
    "freq_dir": safe_load("residual_stream", "base", "02_freq_direction_r2.csv"),
    # NB03 Logit lens
    "convergence": safe_load("logit_lens", "base", "03_convergence_layers.csv"),
    # NB04 Attention
    "head_roles": safe_load("attention", "base", "04_head_role_classification.csv"),
    "attn_entropy": safe_load("attention", "base", "04_attention_entropy.csv"),
    "copy_scores": safe_load("attention", "base", "04_copy_scores.csv"),
    "bos_attn": safe_load("attention", "base", "04_bos_attention.csv"),
    "head_by_band": safe_load("attention", "base", "04_head_role_by_band.csv"),
    # NB05 MLP
    "mlp_contrib": safe_load("mlp", "base", "05_mlp_logit_contribution.csv"),
    "mlp_frac": safe_load("mlp", "base", "05_mlp_contribution_fraction.csv"),
    "neuron_sel": safe_load("mlp", "base", "05_neuron_selectivity_summary.csv"),
    "mlp_sparsity": safe_load("mlp", "base", "05_mlp_sparsity.csv"),
    # NB06 Info-theoretic
    "mi_ksg": safe_load("info_theoretic", "base", "06_mi_ksg_trajectory.csv"),
    "mi_probe": safe_load("info_theoretic", "base", "06_mi_probe_trajectory.csv"),
    "delta_mi": safe_load("info_theoretic", "base", "06_delta_mi.csv"),
    "coding_eff": safe_load("info_theoretic", "base", "06_coding_efficiency.csv"),
    "geom_int": safe_load("info_theoretic", "base", "06_geometric_integration.csv"),
    # Circuit comparisons (02c-06c)
    "circ_probe": safe_load(
        "residual_stream", "circuit", "02c_circuit_probe_trajectory.csv"
    ),
    "circ_sep": safe_load(
        "residual_stream", "circuit", "02c_circuit_separation_trajectory.csv"
    ),
    "circ_cka": safe_load("residual_stream", "comparison", "02c_base_circuit_cka.csv"),
    "circ_band_pres": safe_load(
        "residual_stream", "comparison", "02c_per_band_preservation.csv"
    ),
    "circ_summary": safe_load("residual_stream", "circuit", "02c_circuit_summary.csv"),
    "circ_master": safe_load(
        "residual_stream", "circuit", "02c_master_circuit_residual.csv"
    ),
    # 03c logit lens circuit
    "circ_conv": safe_load("logit_lens", "circuit", "03c_circuit_convergence.csv"),
    "circ_conv_comp": safe_load(
        "logit_lens", "comparison", "03c_convergence_comparison.csv"
    ),
    # 04c attention circuit
    "circ_head_stability": safe_load(
        "attention", "comparison", "04c_head_role_stability.csv"
    ),
    "circ_attn_kl": safe_load(
        "attention", "comparison", "04c_attention_kl_divergence.csv"
    ),
    # 05c MLP circuit
    "circ_mlp_contrib": safe_load("mlp", "circuit", "05c_circuit_mlp_contribution.csv"),
    "circ_neuron_corr": safe_load(
        "mlp", "comparison", "05c_neuron_selectivity_correlation.csv"
    ),
    # 06c info circuit
    "circ_mi": safe_load("info_theoretic", "circuit", "06c_circuit_mi_trajectory.csv"),
    "circ_coding": safe_load(
        "info_theoretic", "circuit", "06c_circuit_coding_efficiency.csv"
    ),
}

loaded = {k: len(v) for k, v in data.items() if not v.empty}
print(f"Loaded {len(loaded)}/{len(data)} datasets")
for k, n in sorted(loaded.items()):
    print(f"  {k}: {n} rows")

Loaded 33/37 datasets
  attn_entropy: 10560 rows
  bos_attn: 704 rows
  circ_band_pres: 20 rows
  circ_cka: 870 rows
  circ_coding: 246 rows
  circ_conv: 75 rows
  circ_master: 60 rows
  circ_mi: 246 rows
  circ_mlp_contrib: 870 rows
  circ_probe: 174 rows
  circ_sep: 174 rows
  circ_summary: 4 rows
  cka: 150 rows
  coding_eff: 174 rows
  convergence: 75 rows
  copy_scores: 704 rows
  delta_mi: 174 rows
  dim: 75 rows
  emb_master: 15 rows
  freq_dir: 174 rows
  geom_int: 24 rows
  head_by_band: 3520 rows
  head_roles: 704 rows
  knn: 15 rows
  mi_ksg: 174 rows
  mi_probe: 174 rows
  mlp_contrib: 870 rows
  mlp_frac: 870 rows
  mlp_sparsity: 870 rows
  neuron_sel: 58 rows
  norms: 25 rows
  probe: 174 rows
  separation: 174 rows


## 2. R1: Embedding Structure Tests

- H-R1.1: k-NN purity > chance (permutation test)
- H-R1.2: Linear probe accuracy > chance (binomial test)
- H-R1.3: Embedding norms correlate with frequency rank (Spearman)
- H-R1.4: Separation ratio scales with model size (Spearman)
- H-R1.5: CKA between adjacent bands > distant bands (Mann-Whitney)
- H-R1.6: Intrinsic dimensionality differs by band (Kruskal-Wallis)

In [3]:
# H-R1.1: k-NN purity > chance
df = data["knn"]
if not df.empty and "purity" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        if dm.empty:
            continue
        mean_purity = dm["purity"].mean()
        chance = 1.0 / len(BANDS)
        # One-sample t-test against chance
        t, p = stats.ttest_1samp(dm["purity"], chance)
        d = (mean_purity - chance) / dm["purity"].std() if dm["purity"].std() > 0 else 0
        add_result(
            "R1_embedding",
            "H-R1.1",
            model,
            "purity > chance",
            "t",
            t,
            p / 2 if t > 0 else 1.0,
            d,
            len(dm),
        )
        print(
            f"  H-R1.1 {model}: purity={mean_purity:.3f}, chance={chance:.3f}, t={t:.2f}, p={p / 2:.4f}"
        )
else:
    print("  SKIP H-R1.1: knn_purity not available")

# H-R1.2: Probe accuracy > chance
df = data["emb_master"]
if not df.empty and "probe_accuracy" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        if dm.empty:
            continue
        accs = dm["probe_accuracy"]
        chance = 1.0 / len(BANDS)
        t, p = stats.ttest_1samp(accs, chance)
        d = (accs.mean() - chance) / accs.std() if accs.std() > 0 else 0
        add_result(
            "R1_embedding",
            "H-R1.2",
            model,
            "probe > chance",
            "t",
            t,
            p / 2 if t > 0 else 1.0,
            d,
            len(dm),
        )
        print(f"  H-R1.2 {model}: acc={accs.mean():.3f}, t={t:.2f}, p={p / 2:.4f}")
else:
    print("  SKIP H-R1.2: probe accuracy not available")

# H-R1.3: Embedding norms ~ frequency rank
df = data["norms"]
if not df.empty and "mean_norm" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        if dm.empty or "band" not in dm.columns:
            continue
        band_norms = dm.groupby("band")["mean_norm"].mean()
        ranks = [FREQUENCY_RANK[b] for b in band_norms.index]
        rho, p = stats.spearmanr(ranks, band_norms.values)
        add_result(
            "R1_embedding",
            "H-R1.3",
            model,
            "norm ~ freq_rank",
            "spearman_rho",
            rho,
            p,
            rho,
            len(band_norms),
        )
        print(f"  H-R1.3 {model}: rho={rho:.3f}, p={p:.4f}")
else:
    print("  SKIP H-R1.3: norms not available")

# H-R1.4: Separation ratio scales with model size
df = data["emb_master"]
if not df.empty and "separation_ratio" in df.columns:
    sep_by_model = df.groupby("model")["separation_ratio"].mean()
    caps = [MODEL_CAPACITY[m] for m in sep_by_model.index]
    if len(caps) >= 3:
        rho, p = stats.spearmanr(caps, sep_by_model.values)
        add_result(
            "R1_embedding",
            "H-R1.4",
            "all",
            "sep_ratio ~ model_size",
            "spearman_rho",
            rho,
            p,
            rho,
            len(caps),
        )
        print(f"  H-R1.4 all: rho={rho:.3f}, p={p:.4f}")
else:
    print("  SKIP H-R1.4: separation ratio not available")

# H-R1.5: CKA adjacent > distant
df = data["cka"]
if not df.empty and "is_adjacent" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        adj = dm[dm["is_adjacent"] == True]["cka"].values
        dist = dm[dm["is_adjacent"] == False]["cka"].values
        if len(adj) > 0 and len(dist) > 0:
            U, p = stats.mannwhitneyu(adj, dist, alternative="greater")
            r_bc = U / (len(adj) * len(dist))  # rank-biserial
            add_result(
                "R1_embedding",
                "H-R1.5",
                model,
                "CKA adj > dist",
                "U",
                U,
                p,
                r_bc,
                len(adj) + len(dist),
            )
            print(
                f"  H-R1.5 {model}: adj={np.mean(adj):.3f}, dist={np.mean(dist):.3f}, p={p:.4f}"
            )
else:
    print("  SKIP H-R1.5: CKA not available")

# H-R1.6: Intrinsic dimensionality differs by band
df = data["dim"]
if not df.empty and "mle_dim" in df.columns and "band" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        groups = [
            dm[dm["band"] == b]["mle_dim"].values
            for b in BANDS
            if b in dm["band"].values
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
            n_total = sum(len(g) for g in groups)
            eta2 = (H - len(groups) + 1) / (n_total - len(groups))
            add_result(
                "R1_embedding",
                "H-R1.6",
                model,
                "dim differs by band",
                "H",
                H,
                p,
                eta2,
                n_total,
            )
            print(f"  H-R1.6 {model}: H={H:.2f}, p={p:.4f}, eta2={eta2:.3f}")
else:
    print("  SKIP H-R1.6: dimensionality not available")

print(
    f"\nR1 tests collected: {len([r for r in all_results if r['domain'] == 'R1_embedding'])}"
)

  H-R1.1 pythia-70m: purity=0.315, chance=0.200, t=26.91, p=0.0007
  H-R1.1 pythia-160m: purity=0.311, chance=0.200, t=27.92, p=0.0006
  H-R1.1 pythia-410m: purity=0.318, chance=0.200, t=23.18, p=0.0009
  H-R1.1 pythia-1b: purity=0.326, chance=0.200, t=20.01, p=0.0012
  H-R1.1 pythia-1.4b: purity=0.328, chance=0.200, t=28.40, p=0.0006
  H-R1.2 pythia-70m: acc=0.481, t=28.74, p=0.0006
  H-R1.2 pythia-160m: acc=0.491, t=22.97, p=0.0009
  H-R1.2 pythia-410m: acc=0.499, t=23.80, p=0.0009
  H-R1.2 pythia-1b: acc=0.508, t=21.34, p=0.0011
  H-R1.2 pythia-1.4b: acc=0.521, t=39.28, p=0.0003
  SKIP H-R1.3: norms not available
  H-R1.4 all: rho=1.000, p=0.0000
  H-R1.5 pythia-70m: adj=0.473, dist=0.477, p=0.9156
  H-R1.5 pythia-160m: adj=0.549, dist=0.555, p=0.7706
  H-R1.5 pythia-410m: adj=0.578, dist=0.583, p=0.7832
  H-R1.5 pythia-1b: adj=0.640, dist=0.651, p=0.8941
  H-R1.5 pythia-1.4b: adj=0.634, dist=0.645, p=0.9279
  SKIP H-R1.6: dimensionality not available

R1 tests collected: 16


## 3. R2: Residual Stream Tests

- H-R2.1: Probe accuracy increases through layers (Jonckheere-Terpstra)
- H-R2.2: Separation ratio increases through layers (trend test)
- H-R2.3: Frequency direction R² > null (permutation)
- H-R2.4: Convergence layer differs by band (Kruskal-Wallis)
- H-R2.5: Low-freq bands converge later (Mann-Whitney)

In [4]:
def jonckheere_terpstra(groups):
    """One-sided Jonckheere-Terpstra trend test (increasing)."""
    J = 0
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            for xi in groups[i]:
                for xj in groups[j]:
                    if xj > xi:
                        J += 1
                    elif xj == xi:
                        J += 0.5
    # Normal approximation
    ns = [len(g) for g in groups]
    N = sum(ns)
    E_J = (N**2 - sum(n**2 for n in ns)) / 4
    V_J = (N**2 * (2 * N + 3) - sum(n**2 * (2 * n + 3) for n in ns)) / 72
    if V_J <= 0:
        return 0, 1.0
    Z = (J - E_J) / np.sqrt(V_J)
    p = 1 - stats.norm.cdf(Z)
    return Z, p


# H-R2.1: Probe accuracy increases through layers
df = data["probe"]
if not df.empty and "accuracy" in df.columns and "layer" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        if dm.empty:
            continue
        layers = sorted(dm["layer"].unique())
        groups = [dm[dm["layer"] == l]["accuracy"].values for l in layers]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 3:
            Z, p = jonckheere_terpstra(groups)
            add_result(
                "R2_residual",
                "H-R2.1",
                model,
                "probe increases w/ layer",
                "JT_Z",
                Z,
                p,
                None,
                sum(len(g) for g in groups),
            )
            print(f"  H-R2.1 {model}: JT Z={Z:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R2.1: probe trajectory not available")

# H-R2.2: Separation ratio increases through layers
df = data["separation"]
if not df.empty and "separation_ratio" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        if dm.empty:
            continue
        layers = sorted(dm["layer"].unique())
        groups = [dm[dm["layer"] == l]["separation_ratio"].values for l in layers]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 3:
            Z, p = jonckheere_terpstra(groups)
            add_result(
                "R2_residual",
                "H-R2.2",
                model,
                "sep_ratio increases w/ layer",
                "JT_Z",
                Z,
                p,
                None,
                sum(len(g) for g in groups),
            )
            print(f"  H-R2.2 {model}: JT Z={Z:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R2.2: separation not available")

# H-R2.3: Frequency direction R2 > null
df = data["freq_dir"]
if not df.empty and "r_squared" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        if dm.empty:
            continue
        r2_vals = dm["r_squared"].values
        mean_r2 = r2_vals.mean()
        t, p = stats.ttest_1samp(r2_vals, 0) if len(r2_vals) > 1 else (np.nan, np.nan)
        add_result(
            "R2_residual",
            "H-R2.3",
            model,
            "freq_dir R2 > 0",
            "t",
            t,
            p / 2 if not np.isnan(t) and t > 0 else p,
            mean_r2,
            len(r2_vals),
        )
        print(f"  H-R2.3 {model}: R2={mean_r2:.3f}, p={p:.4f}")
else:
    print("  SKIP H-R2.3: freq direction R2 not available")

# H-R2.4: Convergence layer differs by band
df = data["convergence"]
if not df.empty and "mean_convergence_layer" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        groups = [dm[dm["band"] == b]["mean_convergence_layer"].values for b in BANDS]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
            n_total = sum(len(g) for g in groups)
            eta2 = (H - len(groups) + 1) / (n_total - len(groups))
            add_result(
                "R2_residual",
                "H-R2.4",
                model,
                "convergence differs by band",
                "H",
                H,
                p,
                eta2,
                n_total,
            )
            print(f"  H-R2.4 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R2.4: convergence not available")

# H-R2.5: Low-freq bands converge later
if not df.empty and "mean_convergence_layer" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        low = dm[dm["band"].isin(LOW_FREQ_BANDS)]["mean_convergence_layer"].values
        high = dm[dm["band"].isin(HIGH_FREQ_BANDS)]["mean_convergence_layer"].values
        if len(low) > 0 and len(high) > 0:
            U, p = stats.mannwhitneyu(low, high, alternative="greater")
            r_bc = U / (len(low) * len(high))
            add_result(
                "R2_residual",
                "H-R2.5",
                model,
                "low-freq converge later",
                "U",
                U,
                p,
                r_bc,
                len(low) + len(high),
            )
            print(
                f"  H-R2.5 {model}: low={np.mean(low):.2f}, high={np.mean(high):.2f}, p={p:.4f}"
            )

print(
    f"\nR2 tests collected: {len([r for r in all_results if r['domain'] == 'R2_residual'])}"
)

  H-R2.1 pythia-70m: JT Z=2.27, p=0.0116
  H-R2.1 pythia-160m: JT Z=4.03, p=0.0000
  H-R2.1 pythia-410m: JT Z=9.37, p=0.0000
  H-R2.1 pythia-1b: JT Z=6.16, p=0.0000
  H-R2.2 pythia-70m: JT Z=-0.89, p=0.8120
  H-R2.2 pythia-160m: JT Z=5.01, p=0.0000
  H-R2.2 pythia-410m: JT Z=9.00, p=0.0000
  H-R2.2 pythia-1b: JT Z=4.17, p=0.0000
  SKIP H-R2.3: freq direction R2 not available
  H-R2.4 pythia-70m: H=12.90, p=0.0118
  H-R2.4 pythia-160m: H=12.85, p=0.0120
  H-R2.4 pythia-410m: H=13.50, p=0.0091
  H-R2.4 pythia-1b: H=13.52, p=0.0090
  H-R2.4 pythia-1.4b: H=13.50, p=0.0091
  H-R2.5 pythia-70m: low=4.68, high=4.42, p=0.0025
  H-R2.5 pythia-160m: low=9.39, high=8.81, p=0.0011
  H-R2.5 pythia-410m: low=20.57, high=18.21, p=0.0011
  H-R2.5 pythia-1b: low=12.20, high=11.21, p=0.0025
  H-R2.5 pythia-1.4b: low=20.47, high=15.70, p=0.0011

R2 tests collected: 18


## 4. R3: Attention Pattern Tests

- H-R3.1: Induction heads exist and are consistent across draws
- H-R3.2: Attention entropy differs by band (Kruskal-Wallis)
- H-R3.3: Copy scores differ by band (Kruskal-Wallis)
- H-R3.4: BOS-sink attention correlates with frequency rank (Spearman)
- H-R3.5: Head role distribution differs by band (chi-squared)

In [5]:
# H-R3.1: Induction heads exist
df = data["head_roles"]
if not df.empty and "role" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        if dm.empty:
            continue
        n_induction = (dm["role"] == "induction").sum()
        n_total = len(dm)
        frac = n_induction / n_total if n_total > 0 else 0
        # Binomial test: any induction heads found?
        p = (
            stats.binomtest(n_induction, n_total, 0.01, alternative="greater").pvalue
            if n_total > 0
            else 1.0
        )
        add_result(
            "R3_attention",
            "H-R3.1",
            model,
            "induction heads exist",
            "binomial",
            frac,
            p,
            frac,
            n_total,
        )
        print(
            f"  H-R3.1 {model}: {n_induction}/{n_total} induction heads ({frac:.1%}), p={p:.4f}"
        )
else:
    print("  SKIP H-R3.1: head roles not available")

# H-R3.2: Attention entropy differs by band
df = data["attn_entropy"]
if not df.empty and "entropy" in df.columns and "band" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        groups = [
            dm[dm["band"] == b]["entropy"].values
            for b in BANDS
            if b in dm["band"].values
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
            n_total = sum(len(g) for g in groups)
            eta2 = (H - len(groups) + 1) / (n_total - len(groups))
            add_result(
                "R3_attention",
                "H-R3.2",
                model,
                "entropy differs by band",
                "H",
                H,
                p,
                eta2,
                n_total,
            )
            print(f"  H-R3.2 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R3.2: attention entropy not available")

# H-R3.3: Copy scores differ by band
df = data["copy_scores"]
if not df.empty and "copy_score" in df.columns and "band" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        groups = [
            dm[dm["band"] == b]["copy_score"].values
            for b in BANDS
            if b in dm["band"].values
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
            n_total = sum(len(g) for g in groups)
            eta2 = (H - len(groups) + 1) / (n_total - len(groups))
            add_result(
                "R3_attention",
                "H-R3.3",
                model,
                "copy score differs by band",
                "H",
                H,
                p,
                eta2,
                n_total,
            )
            print(f"  H-R3.3 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R3.3: copy scores not available")

# H-R3.4: BOS-sink attention ~ frequency rank
df = data["bos_attn"]
if not df.empty and "band" in df.columns:
    bos_col = [c for c in df.columns if "bos" in c.lower() and c != "band"]
    if bos_col:
        bos_col = bos_col[0]
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            band_vals = dm.groupby("band")[bos_col].mean()
            ranks = [
                FREQUENCY_RANK[b]
                for b in band_vals.index
                if FREQUENCY_RANK.get(b) is not None
            ]
            vals = [
                band_vals[b]
                for b in band_vals.index
                if FREQUENCY_RANK.get(b) is not None
            ]
            if len(ranks) >= 3:
                rho, p = stats.spearmanr(ranks, vals)
                add_result(
                    "R3_attention",
                    "H-R3.4",
                    model,
                    "BOS attn ~ freq_rank",
                    "spearman_rho",
                    rho,
                    p,
                    rho,
                    len(ranks),
                )
                print(f"  H-R3.4 {model}: rho={rho:.3f}, p={p:.4f}")
    else:
        print("  SKIP H-R3.4: no BOS column found")
else:
    print("  SKIP H-R3.4: BOS attention not available")

# H-R3.5: Head role distribution differs by band
df = data["head_by_band"]
if not df.empty and "band" in df.columns and "role" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        ct = pd.crosstab(dm["band"], dm["role"])
        if ct.shape[0] >= 2 and ct.shape[1] >= 2:
            chi2, p, dof, _ = stats.chi2_contingency(ct)
            n_total = ct.values.sum()
            v = np.sqrt(chi2 / (n_total * (min(ct.shape) - 1)))
            add_result(
                "R3_attention",
                "H-R3.5",
                model,
                "role dist differs by band",
                "chi2",
                chi2,
                p,
                v,
                n_total,
            )
            print(f"  H-R3.5 {model}: chi2={chi2:.2f}, p={p:.4f}, V={v:.3f}")
else:
    print("  SKIP H-R3.5: head_role_by_band not available")

print(
    f"\nR3 tests collected: {len([r for r in all_results if r['domain'] == 'R3_attention'])}"
)

  H-R3.1 pythia-70m: 3/48 induction heads (6.2%), p=0.0124
  H-R3.1 pythia-160m: 5/144 induction heads (3.5%), p=0.0153
  H-R3.1 pythia-410m: 8/384 induction heads (2.1%), p=0.0413
  H-R3.1 pythia-1b: 4/128 induction heads (3.1%), p=0.0403
  SKIP H-R3.2: attention entropy not available
  SKIP H-R3.3: copy scores not available
  SKIP H-R3.4: BOS attention not available


  H-R3.5 pythia-70m: chi2=0.72, p=1.0000, V=0.032
  H-R3.5 pythia-160m: chi2=0.92, p=1.0000, V=0.021


  H-R3.5 pythia-410m: chi2=0.58, p=1.0000, V=0.010
  H-R3.5 pythia-1b: chi2=1.13, p=1.0000, V=0.024

R3 tests collected: 8


## 5. R4: MLP Tests

- H-R4.1: MLP logit contribution differs by band (Kruskal-Wallis)
- H-R4.2: MLP fraction shows frequency trend (Jonckheere-Terpstra)
- H-R4.3: Selective neurons scale with model size (Spearman)
- H-R4.4: Sparsity differs by band (Kruskal-Wallis)

In [6]:
# H-R4.1: MLP logit contribution differs by band
df = data["mlp_contrib"]
if not df.empty and "band" in df.columns:
    val_col = [c for c in df.columns if "contrib" in c.lower() or "logit" in c.lower()]
    val_col = (
        [c for c in val_col if c != "band" and c != "model"][0] if val_col else None
    )
    if val_col:
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            groups = [
                dm[dm["band"] == b][val_col].values
                for b in BANDS
                if b in dm["band"].values
            ]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) >= 2:
                H, p = stats.kruskal(*groups)
                n_total = sum(len(g) for g in groups)
                eta2 = (H - len(groups) + 1) / (n_total - len(groups))
                add_result(
                    "R4_mlp",
                    "H-R4.1",
                    model,
                    "MLP contrib differs by band",
                    "H",
                    H,
                    p,
                    eta2,
                    n_total,
                )
                print(f"  H-R4.1 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R4.1: MLP contribution not available")

# H-R4.2: MLP fraction shows frequency trend
df = data["mlp_frac"]
if not df.empty and "band" in df.columns:
    val_col = [c for c in df.columns if "frac" in c.lower()]
    val_col = (
        [c for c in val_col if c != "band" and c != "model"][0] if val_col else None
    )
    if val_col:
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            ordered_bands = [b for b in BANDS if b in dm["band"].values]
            groups = [dm[dm["band"] == b][val_col].values for b in ordered_bands]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) >= 3:
                Z, p = jonckheere_terpstra(groups)
                add_result(
                    "R4_mlp",
                    "H-R4.2",
                    model,
                    "MLP frac freq trend",
                    "JT_Z",
                    Z,
                    p,
                    None,
                    sum(len(g) for g in groups),
                )
                print(f"  H-R4.2 {model}: JT Z={Z:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R4.2: MLP fraction not available")

# H-R4.3: Selective neurons scale with model size
df = data["neuron_sel"]
if not df.empty:
    sel_col = [
        c for c in df.columns if "selective" in c.lower() or "count" in c.lower()
    ]
    if sel_col and "model" in df.columns:
        sel_col = sel_col[0]
        sel_by_model = df.groupby("model")[sel_col].mean()
        caps = [MODEL_CAPACITY[m] for m in sel_by_model.index if m in MODEL_CAPACITY]
        vals = [sel_by_model[m] for m in sel_by_model.index if m in MODEL_CAPACITY]
        if len(caps) >= 3:
            rho, p = stats.spearmanr(caps, vals)
            add_result(
                "R4_mlp",
                "H-R4.3",
                "all",
                "selective neurons ~ model_size",
                "spearman_rho",
                rho,
                p,
                rho,
                len(caps),
            )
            print(f"  H-R4.3 all: rho={rho:.3f}, p={p:.4f}")
else:
    print("  SKIP H-R4.3: neuron selectivity not available")

# H-R4.4: Sparsity differs by band
df = data["mlp_sparsity"]
if not df.empty and "band" in df.columns:
    sp_col = [c for c in df.columns if "spars" in c.lower()]
    sp_col = [c for c in sp_col if c != "band" and c != "model"][0] if sp_col else None
    if sp_col:
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            groups = [
                dm[dm["band"] == b][sp_col].values
                for b in BANDS
                if b in dm["band"].values
            ]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) >= 2:
                H, p = stats.kruskal(*groups)
                n_total = sum(len(g) for g in groups)
                eta2 = (H - len(groups) + 1) / (n_total - len(groups))
                add_result(
                    "R4_mlp",
                    "H-R4.4",
                    model,
                    "sparsity differs by band",
                    "H",
                    H,
                    p,
                    eta2,
                    n_total,
                )
                print(f"  H-R4.4 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R4.4: sparsity not available")

print(
    f"\nR4 tests collected: {len([r for r in all_results if r['domain'] == 'R4_mlp'])}"
)

  H-R4.1 pythia-70m: H=0.39, p=0.9835
  H-R4.1 pythia-160m: H=0.17, p=0.9964
  H-R4.1 pythia-410m: H=12.44, p=0.0143


  H-R4.1 pythia-1b: H=16.35, p=0.0026
  H-R4.2 pythia-70m: JT Z=-3.41, p=0.9997
  H-R4.2 pythia-160m: JT Z=0.65, p=0.2564
  H-R4.2 pythia-410m: JT Z=-0.65, p=0.7429
  H-R4.2 pythia-1b: JT Z=0.39, p=0.3467
  H-R4.3 all: rho=1.000, p=0.0000

R4 tests collected: 9


## 6. R5: Information-Theoretic Tests

- H-R5.1: MI increases through layers (Jonckheere-Terpstra)
- H-R5.2: Delta-MI peak layer differs across models (Kruskal-Wallis)
- H-R5.3: Coding efficiency increases with model size (Spearman)
- H-R5.4: MI correlates with geometric separation (Spearman)

In [7]:
# H-R5.1: MI increases through layers
# Use probe-based MI (primary metric) instead of KSG MI.
# KSG MI fails in high dimensions (d=512-2048, N=1125) and produces
# spurious decreasing trajectories. Probe-MI is a reliable lower bound.
df = data.get("mi_probe", pd.DataFrame())
if df.empty:
    df = data["mi_ksg"]  # Fallback to KSG if probe not available
    mi_col_name = "mi_ksg"
    print("  NOTE: Using KSG MI (probe MI not available)")
else:
    mi_col_name = "mi_probe"
    print("  NOTE: Using probe-based MI (primary metric)")
if not df.empty and "layer" in df.columns:
    mi_col = mi_col_name if mi_col_name in df.columns else None
    if mi_col is None:
        mi_col = [
            c
            for c in df.columns
            if "mi" in c.lower() and c not in ("model", "band", "draw", "layer")
        ]
        mi_col = mi_col[0] if mi_col else None
    if mi_col:
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            if dm.empty:
                continue
            layers = sorted(dm["layer"].unique())
            groups = [dm[dm["layer"] == l][mi_col].values for l in layers]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) >= 3:
                Z, p = jonckheere_terpstra(groups)
                add_result(
                    "R5_info",
                    "H-R5.1",
                    model,
                    "MI increases w/ layer",
                    "JT_Z",
                    Z,
                    p,
                    None,
                    sum(len(g) for g in groups),
                )
                print(
                    f"  H-R5.1 {model}: JT Z={Z:.2f}, p={p:.4f} (metric: {mi_col_name})"
                )
else:
    print("  SKIP H-R5.1: MI trajectory not available")

# H-R5.2: Delta-MI peak layer differs across models
df = data["delta_mi"]
if not df.empty and "layer" in df.columns:
    dmi_col = [c for c in df.columns if "delta" in c.lower() or "mi" in c.lower()]
    dmi_col = [c for c in dmi_col if c not in ("model", "band", "draw", "layer")]
    dmi_col = dmi_col[0] if dmi_col else None
    if dmi_col and "model" in df.columns:
        peak_layers = df.loc[df.groupby(["model", "draw"])[dmi_col].idxmax()]["layer"]
        groups_by_model = []
        for model in MODELS:
            dm = df[df["model"] == model]
            peaks = dm.loc[dm.groupby("draw")[dmi_col].idxmax()]["layer"].values
            if len(peaks) > 0:
                groups_by_model.append(peaks)
        if len(groups_by_model) >= 2:
            H, p = stats.kruskal(*groups_by_model)
            n_total = sum(len(g) for g in groups_by_model)
            eta2 = (H - len(groups_by_model) + 1) / (n_total - len(groups_by_model))
            add_result(
                "R5_info",
                "H-R5.2",
                "all",
                "delta-MI peak differs by model",
                "H",
                H,
                p,
                eta2,
                n_total,
            )
            print(f"  H-R5.2 all: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-R5.2: delta-MI not available")

# H-R5.3: Coding efficiency ~ model size
df = data["coding_eff"]
if not df.empty and "model" in df.columns:
    eff_col = [c for c in df.columns if "effic" in c.lower() or "coding" in c.lower()]
    eff_col = [c for c in eff_col if c not in ("model", "band", "draw", "layer")]
    eff_col = eff_col[0] if eff_col else None
    if eff_col:
        eff_by_model = df.groupby("model")[eff_col].mean()
        caps = [MODEL_CAPACITY[m] for m in eff_by_model.index if m in MODEL_CAPACITY]
        vals = [eff_by_model[m] for m in eff_by_model.index if m in MODEL_CAPACITY]
        if len(caps) >= 3:
            rho, p = stats.spearmanr(caps, vals)
            add_result(
                "R5_info",
                "H-R5.3",
                "all",
                "coding eff ~ model_size",
                "spearman_rho",
                rho,
                p,
                rho,
                len(caps),
            )
            print(f"  H-R5.3 all: rho={rho:.3f}, p={p:.4f}")
else:
    print("  SKIP H-R5.3: coding efficiency not available")

# H-R5.4: MI correlates with geometric separation
df = data["geom_int"]
if not df.empty:
    corr_col = [
        c
        for c in df.columns
        if "corr" in c.lower() or "rho" in c.lower() or "r" == c.lower()
    ]
    if corr_col:
        corr_col = corr_col[0]
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            if dm.empty:
                continue
            vals = dm[corr_col].values
            t, p = stats.ttest_1samp(vals, 0) if len(vals) > 1 else (np.nan, np.nan)
            add_result(
                "R5_info",
                "H-R5.4",
                model,
                "MI ~ geometric separation",
                "t",
                t,
                p / 2 if not np.isnan(t) and t > 0 else p,
                np.mean(vals),
                len(vals),
            )
            print(f"  H-R5.4 {model}: mean_corr={np.mean(vals):.3f}, p={p:.4f}")
else:
    print("  SKIP H-R5.4: geometric integration not available")

print(
    f"\nR5 tests collected: {len([r for r in all_results if r['domain'] == 'R5_info'])}"
)

  NOTE: Using probe-based MI (primary metric)
  H-R5.1 pythia-70m: JT Z=1.50, p=0.0667 (metric: mi_probe)
  H-R5.1 pythia-160m: JT Z=4.65, p=0.0000 (metric: mi_probe)
  H-R5.1 pythia-410m: JT Z=10.55, p=0.0000 (metric: mi_probe)
  H-R5.1 pythia-1b: JT Z=7.06, p=0.0000 (metric: mi_probe)
  H-R5.2 all: H=nan, p=nan
  H-R5.3 all: rho=-1.000, p=0.0000
  H-R5.4 pythia-70m: mean_corr=nan, p=nan
  H-R5.4 pythia-160m: mean_corr=nan, p=nan
  H-R5.4 pythia-410m: mean_corr=nan, p=nan
  H-R5.4 pythia-1b: mean_corr=nan, p=nan

R5 tests collected: 10


<TMPDIR>/env/lib/python3.12/site-packages/scipy/stats/_stats_py.py:8492: RuntimeWarning: invalid value encountered in scalar divide
  h /= ties


## 7. RC1: Circuit vs Base Representations

- H-RC1.1: Circuit probe accuracy < base (paired t-test)
- H-RC1.2: Circuit separation ratio < base (paired t-test)
- H-RC1.3: CKA(base, circuit) differs by band (Kruskal-Wallis)
- H-RC1.4: Circuit convergence layer >= base (paired t-test)
- H-RC1.5: Circuit MI < base MI per layer (paired t-test)

In [8]:
# H-RC1.1: Circuit probe < base
df_base = data["probe"]
df_circ = data["circ_probe"]
if not df_base.empty and not df_circ.empty:
    for model in MODELS:
        base_m = (
            df_base[df_base["model"] == model]
            if "model" in df_base.columns
            else df_base
        )
        circ_m = (
            df_circ[df_circ["model"] == model]
            if "model" in df_circ.columns
            else df_circ
        )
        if base_m.empty or circ_m.empty:
            continue
        # Match by layer and draw, take mean accuracy
        base_acc = base_m.groupby("layer")["accuracy"].mean()
        circ_acc = circ_m.groupby("layer")["accuracy"].mean()
        common = base_acc.index.intersection(circ_acc.index)
        if len(common) >= 3:
            t, p = stats.ttest_rel(base_acc[common], circ_acc[common])
            d = (base_acc[common].mean() - circ_acc[common].mean()) / base_acc[
                common
            ].std()
            add_result(
                "RC1_circuit_vs_base",
                "H-RC1.1",
                model,
                "circuit probe < base",
                "t",
                t,
                p / 2 if t > 0 else 1.0,
                d,
                len(common),
            )
            print(
                f"  H-RC1.1 {model}: base={base_acc[common].mean():.3f}, "
                f"circ={circ_acc[common].mean():.3f}, t={t:.2f}, p={p / 2:.4f}"
            )
else:
    print("  SKIP H-RC1.1: probe data not available")

# H-RC1.2: Circuit separation < base
df_base = data["separation"]
df_circ = data["circ_sep"]
if not df_base.empty and not df_circ.empty:
    for model in MODELS:
        base_m = (
            df_base[df_base["model"] == model]
            if "model" in df_base.columns
            else df_base
        )
        circ_m = (
            df_circ[df_circ["model"] == model]
            if "model" in df_circ.columns
            else df_circ
        )
        base_vals = base_m.groupby("layer")["separation_ratio"].mean()
        circ_vals = circ_m.groupby("layer")["separation_ratio"].mean()
        common = base_vals.index.intersection(circ_vals.index)
        if len(common) >= 3:
            t, p = stats.ttest_rel(base_vals[common], circ_vals[common])
            d = (base_vals[common].mean() - circ_vals[common].mean()) / base_vals[
                common
            ].std()
            add_result(
                "RC1_circuit_vs_base",
                "H-RC1.2",
                model,
                "circuit sep < base",
                "t",
                t,
                p / 2 if t > 0 else 1.0,
                d,
                len(common),
            )
            print(f"  H-RC1.2 {model}: t={t:.2f}, p={p / 2:.4f}")
else:
    print("  SKIP H-RC1.2: separation data not available")

# H-RC1.3: CKA(base, circuit) differs by band
df = data["circ_cka"]
if not df.empty and "band" in df.columns and "cka_base_circuit" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model]
        groups = [
            dm[dm["band"] == b]["cka_base_circuit"].values
            for b in BANDS
            if b in dm["band"].values
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
            n_total = sum(len(g) for g in groups)
            eta2 = (H - len(groups) + 1) / (n_total - len(groups))
            add_result(
                "RC1_circuit_vs_base",
                "H-RC1.3",
                model,
                "CKA differs by band",
                "H",
                H,
                p,
                eta2,
                n_total,
            )
            print(f"  H-RC1.3 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-RC1.3: circuit CKA not available")

# H-RC1.4: Circuit convergence >= base
df_base = data["convergence"]
df_circ = data["circ_conv"]
if not df_base.empty and not df_circ.empty:
    for model in MODELS:
        base_m = df_base[df_base["model"] == model]
        circ_m = (
            df_circ[df_circ["model"] == model]
            if "model" in df_circ.columns
            else df_circ
        )
        if base_m.empty or circ_m.empty:
            continue
        base_conv = base_m.groupby("band")["mean_convergence_layer"].mean()
        circ_conv_col = [c for c in circ_m.columns if "convergence" in c.lower()]
        if circ_conv_col:
            circ_conv = circ_m.groupby("band")[circ_conv_col[0]].mean()
            common = base_conv.index.intersection(circ_conv.index)
            if len(common) >= 3:
                t, p = stats.ttest_rel(circ_conv[common], base_conv[common])
                add_result(
                    "RC1_circuit_vs_base",
                    "H-RC1.4",
                    model,
                    "circuit conv >= base",
                    "t",
                    t,
                    p / 2 if t > 0 else 1.0,
                    None,
                    len(common),
                )
                print(f"  H-RC1.4 {model}: t={t:.2f}, p={p / 2:.4f}")
else:
    print("  SKIP H-RC1.4: convergence data not available")

# H-RC1.5: Circuit MI < base MI
df_base = data["mi_ksg"]
df_circ = data["circ_mi"]
if not df_base.empty and not df_circ.empty:
    mi_col_b = [
        c
        for c in df_base.columns
        if "mi" in c.lower() and c not in ("model", "band", "draw", "layer")
    ]
    mi_col_c = [
        c
        for c in df_circ.columns
        if "mi" in c.lower() and c not in ("model", "band", "draw", "layer")
    ]
    if mi_col_b and mi_col_c:
        for model in MODELS:
            base_m = (
                df_base[df_base["model"] == model]
                if "model" in df_base.columns
                else df_base
            )
            circ_m = (
                df_circ[df_circ["model"] == model]
                if "model" in df_circ.columns
                else df_circ
            )
            base_mi = base_m.groupby("layer")[mi_col_b[0]].mean()
            circ_mi = circ_m.groupby("layer")[mi_col_c[0]].mean()
            common = base_mi.index.intersection(circ_mi.index)
            if len(common) >= 3:
                t, p = stats.ttest_rel(base_mi[common], circ_mi[common])
                add_result(
                    "RC1_circuit_vs_base",
                    "H-RC1.5",
                    model,
                    "circuit MI < base",
                    "t",
                    t,
                    p / 2 if t > 0 else 1.0,
                    None,
                    len(common),
                )
                print(f"  H-RC1.5 {model}: t={t:.2f}, p={p / 2:.4f}")
else:
    print("  SKIP H-RC1.5: MI data not available")

print(
    f"\nRC1 tests collected: {len([r for r in all_results if r['domain'] == 'RC1_circuit_vs_base'])}"
)

  H-RC1.1 pythia-70m: base=0.563, circ=0.583, t=-2.28, p=0.0356
  H-RC1.1 pythia-160m: base=0.601, circ=0.631, t=-4.28, p=0.0007
  H-RC1.1 pythia-410m: base=0.596, circ=0.639, t=-5.77, p=0.0000
  H-RC1.1 pythia-1b: base=0.613, circ=0.707, t=-13.04, p=0.0000


  H-RC1.2 pythia-70m: t=-1.78, p=0.0677
  H-RC1.2 pythia-160m: t=-0.83, p=0.2127
  H-RC1.2 pythia-410m: t=-9.81, p=0.0000
  H-RC1.2 pythia-1b: t=-8.28, p=0.0000
  H-RC1.3 pythia-70m: H=12.10, p=0.0166
  H-RC1.3 pythia-160m: H=10.45, p=0.0335
  H-RC1.3 pythia-410m: H=18.56, p=0.0010


  H-RC1.3 pythia-1b: H=12.04, p=0.0170
  H-RC1.4 pythia-70m: t=-1.22, p=0.1439
  H-RC1.4 pythia-160m: t=-0.15, p=0.4438
  H-RC1.4 pythia-410m: t=-2.49, p=0.0337
  H-RC1.4 pythia-1b: t=-2.68, p=0.0277


  H-RC1.4 pythia-1.4b: t=-2.85, p=0.0232
  H-RC1.5 pythia-70m: t=1.76, p=0.0690
  H-RC1.5 pythia-160m: t=3.62, p=0.0020
  H-RC1.5 pythia-410m: t=-3.27, p=0.0017
  H-RC1.5 pythia-1b: t=-1.04, p=0.1569

RC1 tests collected: 21


## 8. RC2: Circuit Band-Dependence

- H-RC2.1: Representation loss (1 - CKA) differs by band (Kruskal-Wallis)
- H-RC2.2: Low-frequency bands lose more structure under ablation (Mann-Whitney)
- H-RC2.3: Head role stability under ablation differs by band (Kruskal-Wallis)
- H-RC2.4: Neuron selectivity preservation differs by band (Kruskal-Wallis)

In [9]:
# H-RC2.1: Representation loss differs by band
df = data["circ_band_pres"]
if not df.empty and "band" in df.columns and "mean_cka" in df.columns:
    df["repr_loss"] = 1 - df["mean_cka"]
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        groups = [
            dm[dm["band"] == b]["repr_loss"].values
            for b in BANDS
            if b in dm["band"].values
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
            n_total = sum(len(g) for g in groups)
            eta2 = (H - len(groups) + 1) / (n_total - len(groups))
            add_result(
                "RC2_band_dep",
                "H-RC2.1",
                model,
                "repr loss differs by band",
                "H",
                H,
                p,
                eta2,
                n_total,
            )
            print(f"  H-RC2.1 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-RC2.1: band preservation not available")

# H-RC2.2: Low-freq lose more
if not df.empty and "band" in df.columns and "repr_loss" in df.columns:
    for model in MODELS:
        dm = df[df["model"] == model] if "model" in df.columns else df
        low = dm[dm["band"].isin(LOW_FREQ_BANDS)]["repr_loss"].values
        high = dm[dm["band"].isin(HIGH_FREQ_BANDS)]["repr_loss"].values
        if len(low) > 0 and len(high) > 0:
            U, p = stats.mannwhitneyu(low, high, alternative="greater")
            r_bc = U / (len(low) * len(high))
            add_result(
                "RC2_band_dep",
                "H-RC2.2",
                model,
                "low-freq lose more",
                "U",
                U,
                p,
                r_bc,
                len(low) + len(high),
            )
            print(
                f"  H-RC2.2 {model}: low={np.mean(low):.3f}, high={np.mean(high):.3f}, p={p:.4f}"
            )
else:
    print("  SKIP H-RC2.2: repr loss not available")

# H-RC2.3: Head role stability differs by band
df = data["circ_head_stability"]
if not df.empty and "band" in df.columns:
    stab_col = [c for c in df.columns if "stab" in c.lower() or "match" in c.lower()]
    stab_col = (
        [c for c in stab_col if c != "band" and c != "model"][0] if stab_col else None
    )
    if stab_col:
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            groups = [
                dm[dm["band"] == b][stab_col].values
                for b in BANDS
                if b in dm["band"].values
            ]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) >= 2:
                H, p = stats.kruskal(*groups)
                n_total = sum(len(g) for g in groups)
                eta2 = (H - len(groups) + 1) / (n_total - len(groups))
                add_result(
                    "RC2_band_dep",
                    "H-RC2.3",
                    model,
                    "head stability differs by band",
                    "H",
                    H,
                    p,
                    eta2,
                    n_total,
                )
                print(f"  H-RC2.3 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-RC2.3: head stability not available")

# H-RC2.4: Neuron selectivity preservation differs by band
df = data["circ_neuron_corr"]
if not df.empty and "band" in df.columns:
    corr_col = [c for c in df.columns if "corr" in c.lower() or "r" in c.lower()]
    corr_col = [c for c in corr_col if c not in ("model", "band", "draw", "layer")]
    corr_col = corr_col[0] if corr_col else None
    if corr_col:
        for model in MODELS:
            dm = df[df["model"] == model] if "model" in df.columns else df
            groups = [
                dm[dm["band"] == b][corr_col].values
                for b in BANDS
                if b in dm["band"].values
            ]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) >= 2:
                H, p = stats.kruskal(*groups)
                n_total = sum(len(g) for g in groups)
                eta2 = (H - len(groups) + 1) / (n_total - len(groups))
                add_result(
                    "RC2_band_dep",
                    "H-RC2.4",
                    model,
                    "neuron pres differs by band",
                    "H",
                    H,
                    p,
                    eta2,
                    n_total,
                )
                print(f"  H-RC2.4 {model}: H={H:.2f}, p={p:.4f}")
else:
    print("  SKIP H-RC2.4: neuron selectivity correlation not available")

print(
    f"\nRC2 tests collected: {len([r for r in all_results if r['domain'] == 'RC2_band_dep'])}"
)

  H-RC2.1 pythia-70m: H=4.00, p=0.4060
  H-RC2.1 pythia-160m: H=4.00, p=0.4060
  H-RC2.1 pythia-410m: H=4.00, p=0.4060
  H-RC2.1 pythia-1b: H=4.00, p=0.4060
  H-RC2.2 pythia-70m: low=0.002, high=0.002, p=0.6667
  H-RC2.2 pythia-160m: low=0.011, high=0.017, p=0.6667


  H-RC2.2 pythia-410m: low=0.021, high=0.038, p=1.0000
  H-RC2.2 pythia-1b: low=0.060, high=0.035, p=0.6667
  SKIP H-RC2.3: head stability not available
  SKIP H-RC2.4: neuron selectivity correlation not available

RC2 tests collected: 8


<TMPDIR>/ipykernel_378897/3206978741.py:12: RuntimeWarning: invalid value encountered in scalar divide
  eta2 = (H - len(groups) + 1) / (n_total - len(groups))


## 9. RC3: Circuit Scaling

- H-RC3.1: Larger models' circuits preserve more representational structure (Spearman)
- H-RC3.2: Circuit coding efficiency scales with model size (Spearman)
- H-RC3.3: Circuit-base CKA increases with model size (Spearman)

In [10]:
# H-RC3.1: Larger models' circuits preserve more structure
df = data["circ_summary"]
if not df.empty and "circuit_peak_probe_acc" in df.columns:
    caps = [MODEL_CAPACITY[m] for m in df["model"] if m in MODEL_CAPACITY]
    vals = [
        df[df["model"] == m]["circuit_peak_probe_acc"].values[0]
        for m in df["model"]
        if m in MODEL_CAPACITY
    ]
    if len(caps) >= 3:
        rho, p = stats.spearmanr(caps, vals)
        add_result(
            "RC3_scaling",
            "H-RC3.1",
            "all",
            "circuit probe ~ model_size",
            "spearman_rho",
            rho,
            p,
            rho,
            len(caps),
        )
        print(f"  H-RC3.1: rho={rho:.3f}, p={p:.4f}")
else:
    print("  SKIP H-RC3.1: circuit summary not available")

# H-RC3.2: Circuit coding efficiency ~ model size
df = data["circ_coding"]
if not df.empty and "model" in df.columns:
    eff_col = [c for c in df.columns if "effic" in c.lower() or "coding" in c.lower()]
    eff_col = [c for c in eff_col if c not in ("model", "band", "draw", "layer")]
    eff_col = eff_col[0] if eff_col else None
    if eff_col:
        eff_by_model = df.groupby("model")[eff_col].mean()
        caps = [MODEL_CAPACITY[m] for m in eff_by_model.index if m in MODEL_CAPACITY]
        vals = [eff_by_model[m] for m in eff_by_model.index if m in MODEL_CAPACITY]
        if len(caps) >= 3:
            rho, p = stats.spearmanr(caps, vals)
            add_result(
                "RC3_scaling",
                "H-RC3.2",
                "all",
                "circuit coding eff ~ model_size",
                "spearman_rho",
                rho,
                p,
                rho,
                len(caps),
            )
            print(f"  H-RC3.2: rho={rho:.3f}, p={p:.4f}")
else:
    print("  SKIP H-RC3.2: circuit coding efficiency not available")

# H-RC3.3: CKA(base, circuit) ~ model size
df = data["circ_summary"]
if not df.empty and "mean_cka_base_circuit" in df.columns:
    caps = [MODEL_CAPACITY[m] for m in df["model"] if m in MODEL_CAPACITY]
    vals = [
        df[df["model"] == m]["mean_cka_base_circuit"].values[0]
        for m in df["model"]
        if m in MODEL_CAPACITY
    ]
    if len(caps) >= 3:
        rho, p = stats.spearmanr(caps, vals)
        add_result(
            "RC3_scaling",
            "H-RC3.3",
            "all",
            "CKA ~ model_size",
            "spearman_rho",
            rho,
            p,
            rho,
            len(caps),
        )
        print(f"  H-RC3.3: rho={rho:.3f}, p={p:.4f}")
else:
    print("  SKIP H-RC3.3: circuit CKA summary not available")

print(
    f"\nRC3 tests collected: {len([r for r in all_results if r['domain'] == 'RC3_scaling'])}"
)

  H-RC3.1: rho=0.800, p=0.2000
  H-RC3.2: rho=-1.000, p=0.0000
  H-RC3.3: rho=-1.000, p=0.0000

RC3 tests collected: 3


## 10. BH-FDR Correction & Summary

In [11]:
from statsmodels.stats.multitest import multipletests

df_results = pd.DataFrame(all_results)
print(f"Total tests: {len(df_results)}")

if not df_results.empty:
    # Replace NaN p-values with 1.0
    df_results["p_value"] = df_results["p_value"].fillna(1.0)

    # BH-FDR correction
    reject, p_adj, _, _ = multipletests(df_results["p_value"].values, method="fdr_bh")
    df_results["p_adjusted"] = p_adj
    df_results["significant"] = reject

    # Verdict
    def get_verdict(row):
        if row["significant"]:
            return "supported"
        elif row["p_adjusted"] < 0.1:
            return "marginal"
        else:
            return "not supported"

    df_results["verdict"] = df_results.apply(get_verdict, axis=1)

    save_analysis(df_results, "08_all_test_results.csv")
    print(
        f"\nSignificant (FDR < {ALPHA}): {df_results['significant'].sum()}/{len(df_results)}"
    )
    print(
        f"Marginal (0.05-0.10): {((df_results['p_adjusted'] >= ALPHA) & (df_results['p_adjusted'] < 0.1)).sum()}"
    )

    # Domain summary
    domain_summary = (
        df_results.groupby("domain")
        .agg(
            n_tests=("significant", "count"),
            n_significant=("significant", "sum"),
        )
        .reset_index()
    )
    domain_summary["pct_significant"] = (
        domain_summary["n_significant"] / domain_summary["n_tests"] * 100
    )
    print("\nPer-domain summary:")
    print(domain_summary.to_string(index=False))
else:
    print("No test results collected.")

Total tests: 93

Significant (FDR < 0.05): 43/93
Marginal (0.05-0.10): 3

Per-domain summary:
             domain  n_tests  n_significant  pct_significant
       R1_embedding       16             11        68.750000
        R2_residual       18             17        94.444444
       R3_attention        8              2        25.000000
             R4_mlp        9              3        33.333333
            R5_info       10              4        40.000000
RC1_circuit_vs_base       21              4        19.047619
       RC2_band_dep        8              0         0.000000
        RC3_scaling        3              2        66.666667


In [12]:
# Per-hypothesis verdict table
if not df_results.empty:
    hyp_summary = (
        df_results.groupby(["domain", "hypothesis", "comparison"])
        .agg(
            n_models=("model", "nunique"),
            n_significant=("significant", "sum"),
            mean_effect=("effect_size", "mean"),
            min_p_adj=("p_adjusted", "min"),
        )
        .reset_index()
    )

    # Verdict: supported if majority of models significant
    hyp_summary["verdict"] = hyp_summary.apply(
        lambda r: (
            "supported"
            if r["n_significant"] > r["n_models"] / 2
            else ("marginal" if r["n_significant"] > 0 else "not supported")
        ),
        axis=1,
    )

    save_analysis(hyp_summary, "08_hypothesis_verdicts.csv")
    print("Hypothesis verdicts:")
    for _, row in hyp_summary.iterrows():
        marker = (
            "+"
            if row["verdict"] == "supported"
            else ("~" if row["verdict"] == "marginal" else "-")
        )
        print(
            f"  [{marker}] {row['hypothesis']:8s} | {row['verdict']:15s} | "
            f"sig={row['n_significant']:.0f}/{row['n_models']:.0f} | "
            f"effect={row['mean_effect']:.3f} | min_p={row['min_p_adj']:.4f}"
        )

Hypothesis verdicts:
  [+] H-R1.1   | supported       | sig=5/5 | effect=14.597 | min_p=0.0033
  [+] H-R1.2   | supported       | sig=5/5 | effect=15.719 | min_p=0.0020
  [+] H-R1.4   | supported       | sig=1/1 | effect=1.000 | min_p=0.0000
  [-] H-R1.5   | not supported   | sig=0/5 | effect=0.380 | min_p=1.0000
  [+] H-R2.1   | supported       | sig=4/4 | effect=nan | min_p=0.0000
  [+] H-R2.2   | supported       | sig=3/4 | effect=nan | min_p=0.0000
  [+] H-R2.4   | supported       | sig=5/5 | effect=0.926 | min_p=0.0241
  [+] H-R2.5   | supported       | sig=5/5 | effect=1.000 | min_p=0.0038
  [~] H-R3.1   | marginal        | sig=2/4 | effect=0.037 | min_p=0.0295
  [-] H-R3.5   | not supported   | sig=0/4 | effect=0.022 | min_p=1.0000
  [~] H-R4.1   | marginal        | sig=2/4 | effect=0.003 | min_p=0.0075
  [-] H-R4.2   | not supported   | sig=0/4 | effect=nan | min_p=0.4769
  [+] H-R4.3   | supported       | sig=1/1 | effect=1.000 | min_p=0.0000
  [+] H-R5.1   | supported       |

## 11. Visualizations

In [13]:
# Domain-level significance bar chart
if not df_results.empty:
    domain_order = [
        "R1_embedding",
        "R2_residual",
        "R3_attention",
        "R4_mlp",
        "R5_info",
        "RC1_circuit_vs_base",
        "RC2_band_dep",
        "RC3_scaling",
    ]
    domain_labels = [
        "R1: Embed",
        "R2: Resid",
        "R3: Attn",
        "R4: MLP",
        "R5: Info",
        "RC1: Circ/Base",
        "RC2: Band Dep",
        "RC3: Scaling",
    ]

    fig, ax = plt.subplots(figsize=(10, 5))
    ds = domain_summary.set_index("domain")

    present = [d for d in domain_order if d in ds.index]
    labels = [domain_labels[domain_order.index(d)] for d in present]

    x = range(len(present))
    total = [ds.loc[d, "n_tests"] for d in present]
    sig = [ds.loc[d, "n_significant"] for d in present]

    ax.bar(x, total, color="lightgray", label="Total tests")
    ax.bar(x, sig, color="coral", label="Significant (FDR)")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Number of Tests")
    ax.set_title("Hypothesis Testing Summary by Domain")
    ax.legend()
    fig.tight_layout()
    save_figure(fig, "viz_08_01_domain_significance.png")

In [14]:
# Hypothesis verdict heatmap (model x hypothesis)
if not df_results.empty:
    all_models = [m for m in MODELS if m in df_results["model"].values] + (
        ["all"] if "all" in df_results["model"].values else []
    )
    hypotheses = sorted(df_results["hypothesis"].unique())

    verdict_map = {"supported": 1, "marginal": 0.5, "not supported": 0}
    mat = np.full((len(all_models), len(hypotheses)), np.nan)

    for i, model in enumerate(all_models):
        for j, hyp in enumerate(hypotheses):
            subset = df_results[
                (df_results["model"] == model) & (df_results["hypothesis"] == hyp)
            ]
            if not subset.empty:
                mat[i, j] = verdict_map.get(subset.iloc[0]["verdict"], 0)

    fig, ax = plt.subplots(
        figsize=(max(12, len(hypotheses) * 0.6), max(4, len(all_models) * 0.8))
    )
    from matplotlib.colors import ListedColormap

    cmap = ListedColormap(["#d9534f", "#f0ad4e", "#5cb85c"])  # red, yellow, green

    im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(hypotheses)))
    ax.set_xticklabels(hypotheses, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(all_models)))
    ax.set_yticklabels(all_models)
    ax.set_title(
        "Hypothesis Verdicts (green=supported, yellow=marginal, red=not supported)"
    )
    ax.grid(False)
    fig.tight_layout()
    save_figure(fig, "viz_08_02_hypothesis_heatmap.png")

In [15]:
# Circuit vs Base comparison: RC1 effect sizes
rc1 = (
    df_results[df_results["domain"] == "RC1_circuit_vs_base"]
    if not df_results.empty
    else pd.DataFrame()
)
if not rc1.empty:
    fig, ax = plt.subplots(figsize=(10, 5))

    hyps = sorted(rc1["hypothesis"].unique())
    for i, hyp in enumerate(hyps):
        sub = rc1[rc1["hypothesis"] == hyp]
        for _, row in sub.iterrows():
            color = "coral" if row["significant"] else "lightgray"
            ax.barh(
                f"{hyp}\n{row['model']}",
                row["effect_size"],
                color=color,
                edgecolor="white",
            )

    ax.set_xlabel("Effect Size")
    ax.set_title("RC1: Circuit vs Base: Effect Sizes")
    ax.axvline(x=0, color="gray", linestyle="-", linewidth=0.5)
    fig.tight_layout()
    save_figure(fig, "viz_08_03_rc1_effect_sizes.png")

## 12. Export & Final Summary

In [16]:
if not df_results.empty:
    total = len(df_results)
    sig = df_results["significant"].sum()
    marg = (
        (df_results["p_adjusted"] >= ALPHA) & (df_results["p_adjusted"] < 0.1)
    ).sum()

    print(f"=" * 60)
    print(f"NB08 REPRESENTATIONAL INFERENTIAL SUMMARY")
    print(f"=" * 60)
    print(f"Total hypothesis tests: {total}")
    print(f"Significant (FDR < {ALPHA}): {sig} ({sig / total * 100:.1f}%)")
    print(f"Marginal (0.05-0.10): {marg}")
    print(f"Not supported: {total - sig - marg}")
    print()
    print("Base model domains (R1-R5):")
    base_mask = df_results["domain"].str.startswith("R") & ~df_results[
        "domain"
    ].str.startswith("RC")
    print(
        f"  Tests: {base_mask.sum()}, Significant: {df_results[base_mask]['significant'].sum()}"
    )
    print("\nCircuit domains (RC1-RC3):")
    circ_mask = df_results["domain"].str.startswith("RC")
    print(
        f"  Tests: {circ_mask.sum()}, Significant: {df_results[circ_mask]['significant'].sum()}"
    )
    print()
    print("Output files:")
    print(f"  08_all_test_results.csv")
    print(f"  08_hypothesis_verdicts.csv")
    print(f"  viz_08_01_domain_significance.png")
    print(f"  viz_08_02_hypothesis_heatmap.png")
    print(f"  viz_08_03_rc1_effect_sizes.png")
else:
    print("No results to summarize.")

print("\nNB08 complete.")

NB08 REPRESENTATIONAL INFERENTIAL SUMMARY
Total hypothesis tests: 93
Significant (FDR < 0.05): 43 (46.2%)
Marginal (0.05-0.10): 3
Not supported: 47

Base model domains (R1-R5):
  Tests: 61, Significant: 37

Circuit domains (RC1-RC3):
  Tests: 32, Significant: 6

Output files:
  08_all_test_results.csv
  08_hypothesis_verdicts.csv
  viz_08_01_domain_significance.png
  viz_08_02_hypothesis_heatmap.png
  viz_08_03_rc1_effect_sizes.png

NB08 complete.
